# FLUX.2 [klein] LoRA generation

LoRAはBase 4Bで学習し、生成は高速な蒸留4B（4 steps）で行います。低RAMのため生成処理は別プロセスに分離します。

In [ ]:
from pathlib import Path
import subprocess, sys
from IPython.display import display, Image as DisplayImage

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists())
PROMPT_MD = PROJECT_ROOT / 'prompts/prompt1.md'
OUTPUT = PROJECT_ROOT / 'outputs/flux2/generation/notebook.png'
SEED = 42
USE_MARKDOWN_LORA = False  # prompt1.mdのSD1.5 LoRAはFLUX.2で使えない
MEMORY_MODE = 'auto'  # 8GB VRAMではsequential-offloadを自動選択
QUANTIZATION = 'auto'  # 8GB VRAMではbitsandbytes NF4 4bitを自動選択

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / 'flux2/generate.py'),
    '--prompt-md', str(PROMPT_MD),
    '--output', str(OUTPUT),
    '--seed', str(SEED),
    '--memory-mode', MEMORY_MODE,
    '--quantization', QUANTIZATION,
    '--width', '512', '--height', '512',
]
if not USE_MARKDOWN_LORA:
    command.append('--no-lora')
subprocess.run([*command, '--dry-run'], cwd=PROJECT_ROOT, check=True)

In [ ]:
RUN_GENERATION = False
if RUN_GENERATION:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    display(DisplayImage(filename=str(OUTPUT)))
else:
    print('RUN_GENERATION=False: モデルは読み込んでいません')

## 画像編集

編集する場合はCLIへ `--input-image /workspace/path/to/reference.png` を追加します。編集では通常 `--guidance-scale 4.0` を指定します。